# 📰 Fake News Detection — Intern Infobyte (Task 1)
**Author:** Md Chinna Mahaboob  |  **Goal:** classify news as REAL or FAKE

**Pipeline:** Dataset → Preprocessing → Feature Engineering (TF-IDF + Sentiment) → Model Building → Evaluation → Live Prediction

## 1. Install & import libraries
Run the first line once (remove the `#`), then comment it again.

In [ ]:
# !pip install pandas numpy scikit-learn nltk vaderSentiment matplotlib seaborn joblib

import re, os, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix, hstack
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix, classification_report)

try:
    from nltk.stem import PorterStemmer
    stemmer = PorterStemmer()
except ImportError:
    stemmer = None
try:
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    vader = SentimentIntensityAnalyzer()
except ImportError:
    vader = None
print("Stemmer:", stemmer is not None, "| VADER:", vader is not None)

## 2. Dataset selection
Kaggle **Fake and Real News Dataset** → download `Fake.csv` and `True.csv` and put them in a `data/` folder next to this notebook (or the same folder as the notebook).

In [ ]:
def find(name):
    for p in [f"data/{name}", f"../data/{name}", name]:
        if os.path.exists(p):
            return p
    return None

fake_p, true_p = find("Fake.csv"), find("True.csv")
assert fake_p and true_p, "Put Fake.csv and True.csv in the data/ folder!"

fake, true = pd.read_csv(fake_p), pd.read_csv(true_p)
fake["label"], true["label"] = 1, 0          # 1 = FAKE, 0 = REAL
df = pd.concat([fake, true], ignore_index=True)
df["text"] = df["title"].fillna("") + " . " + df["text"].fillna("")
df = df[["text", "label"]].drop_duplicates().sample(frac=1, random_state=42).reset_index(drop=True)

print(df.shape)
df["label"].map({0: "REAL", 1: "FAKE"}).value_counts().plot(kind="bar", color=["seagreen", "crimson"], title="Class balance")
plt.show()
df.head()

## 3. Data preprocessing
Lowercase → remove URLs/punctuation → remove stopwords → stemming. Also removes the `(Reuters)` publisher tag, which would otherwise leak the answer to the model.

In [ ]:
LEAK = re.compile(r"^.{0,60}?\(reuters\)\s*-?\s*", re.IGNORECASE)
URL = re.compile(r"http\S+|www\.\S+")
NON_ALPHA = re.compile(r"[^a-z\s]")

def clean_text(text):
    text = LEAK.sub("", str(text))
    text = NON_ALPHA.sub(" ", URL.sub(" ", text.lower()))
    tokens = [t for t in text.split() if t not in ENGLISH_STOP_WORDS and len(t) > 2]
    if stemmer:
        tokens = [stemmer.stem(t) for t in tokens]
    return " ".join(tokens)

df["clean"] = df["text"].apply(clean_text)
df = df[df["clean"].str.len() > 0]
df[["text", "clean"]].head(3)

## 4. Feature engineering
**TF-IDF** (1–2 word n-grams) + **sentiment** scores (VADER).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df[["text", "clean"]], df["label"], test_size=0.2, stratify=df["label"], random_state=42)

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
T_train = tfidf.fit_transform(X_train["clean"])
T_test = tfidf.transform(X_test["clean"])

def sentiment(texts):
    if vader is None:
        return csr_matrix((len(texts), 4))
    rows = []
    for t in texts:
        s = vader.polarity_scores(str(t)[:2000])
        rows.append([s["neg"], s["neu"], s["pos"], s["compound"]])
    return csr_matrix(np.array(rows))

X_tr = hstack([T_train, sentiment(X_train["text"].values)]).tocsr()
X_te = hstack([T_test, sentiment(X_test["text"].values)]).tocsr()
print("Feature matrix:", X_tr.shape)

## 5. Model building
Three models are trained and compared.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, C=5),
    "Linear SVM": LinearSVC(C=1.0),
    "Naive Bayes": MultinomialNB(alpha=0.1),   # TF-IDF only (needs non-negative features)
}

results, preds = {}, {}
for name, m in models.items():
    a, b = (T_train, T_test) if name == "Naive Bayes" else (X_tr, X_te)
    m.fit(a, y_train)
    p = m.predict(b)
    preds[name] = p
    results[name] = {"Accuracy": accuracy_score(y_test, p), "Precision": precision_score(y_test, p),
                     "Recall": recall_score(y_test, p), "F1": f1_score(y_test, p)}

res = pd.DataFrame(results).T.round(4)
res

## 6. Evaluation
Accuracy, precision, recall, F1-score, confusion matrix.

In [ ]:
res.plot(kind="bar", figsize=(8, 4), ylim=(res.values.min() - 0.02, 1.0), title="Model comparison")
plt.xticks(rotation=15); plt.show()

best = res["F1"].idxmax()
print("Best model:", best)
print(classification_report(y_test, preds[best], target_names=["REAL", "FAKE"]))

sns.heatmap(confusion_matrix(y_test, preds[best]), annot=True, fmt="d", cmap="Blues",
            xticklabels=["REAL", "FAKE"], yticklabels=["REAL", "FAKE"])
plt.title(f"Confusion matrix - {best}"); plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.show()

### Which words push a prediction towards FAKE or REAL?

In [ ]:
lr = models["Logistic Regression"]
names = np.array(list(tfidf.get_feature_names_out()) + ["sent_neg", "sent_neu", "sent_pos", "sent_compound"])
coef = lr.coef_.ravel(); order = np.argsort(coef)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].barh(names[order[-15:]], coef[order[-15:]], color="crimson"); ax[0].set_title("Top FAKE indicators")
ax[1].barh(names[order[:15]], coef[order[:15]], color="seagreen"); ax[1].set_title("Top REAL indicators")
plt.tight_layout(); plt.show()

## 7. Live prediction (use this part in your demo video 🎥)

In [ ]:
def predict(text):
    vec = tfidf.transform([clean_text(text)])
    X = hstack([vec, sentiment([text])]).tocsr()
    p_fake = lr.predict_proba(X)[0][1]
    label = "FAKE" if p_fake >= 0.5 else "REAL"
    conf = p_fake if p_fake >= 0.5 else 1 - p_fake
    print(f"Prediction: {label}  (confidence {conf:.1%})")

predict("PASTE A NEWS HEADLINE OR ARTICLE HERE")

### Interactive mode — type any news text, enter `q` to stop

In [ ]:
while True:
    text = input("Paste news headline/article (q to quit): ").strip()
    if text.lower() in ("q", "quit", "exit"):
        break
    if len(text.split()) < 3:
        print("Please enter a longer text.")
        continue
    predict(text)

## 8. Save the model

In [ ]:
os.makedirs("models", exist_ok=True)
joblib.dump({"tfidf": tfidf, "model": lr}, "models/fake_news_model.joblib")
res.to_csv("metrics.csv")
print("Saved!")

## 9. Conclusion
- Best model: _(write here)_ with F1 = _(write here)_
- Limitations: dataset is mostly US political news (2016-2018); the model learns writing style, not truth.
- Future work: LSTM / BERT, Liar dataset, LIME/SHAP explainability.